# VSD Edge AI - Session 2: Classification Models & Embedded Deployment
This notebook covers:
1. Dataset Exploration (`Social_Network_Ads.csv`)
2. Feature Scaling (StandardScaler)
3. Logistic Regression Classifier
4. K-Nearest Neighbors (KNN) Classifier (Scaled vs Unscaled)
5. Support Vector Machine (Linear SVM)
6. C Header Generation (`scaler.h`, `svm_model.h`) for RISC-V VSDSquadron Pro

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from matplotlib.colors import ListedColormap


## 1. Load and Inspect Dataset

In [2]:
dataset = pd.read_csv('Social_Network_Ads.csv')
print(dataset.head())
X = dataset[['Age', 'EstimatedSalary']].values
y = dataset['Purchased'].values

    User ID  Gender  Age  EstimatedSalary  Purchased
0  15624510    Male   19            19000          0
1  15810944    Male   35            20000          0
2  15668575  Female   26            43000          0
3  15603246  Female   27            57000          0
4  15804002    Male   19            76000          0


## 2. Train-Test Split and Feature Scaling

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

print(f"Mean: {sc.mean_}")
print(f"Scale: {sc.scale_}")

Mean: [3.81266667e+01 6.95833333e+04]
Scale: [1.00977203e+01 3.44909127e+04]


## 3. Logistic Regression

In [4]:
lr_classifier = LogisticRegression(random_state=0)
lr_classifier.fit(X_train_sc, y_train)
y_pred_lr = lr_classifier.predict(X_test_sc)

print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print(f"Coefficients: {lr_classifier.coef_}, Intercept: {lr_classifier.intercept_}")

Logistic Regression Accuracy: 89.00%
Confusion Matrix:
[[65  3]
 [ 8 24]]
Coefficients: [[2.07665837 1.11008221]], Intercept: [-0.95217247]


## 4. K-Nearest Neighbors (KNN)

In [5]:
knn_classifier = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
knn_classifier.fit(X_train_sc, y_train)
y_pred_knn = knn_classifier.predict(X_test_sc)

print(f"KNN (Scaled) Accuracy: {accuracy_score(y_test, y_pred_knn)*100:.2f}%")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))

KNN (Scaled) Accuracy: 93.00%
Confusion Matrix:
[[64  4]
 [ 3 29]]


## 5. Support Vector Machine (Linear SVM)

In [6]:
svm_classifier = SVC(kernel='linear', random_state=0)
svm_classifier.fit(X_train_sc, y_train)
y_pred_svm = svm_classifier.predict(X_test_sc)

print(f"Linear SVM Accuracy: {accuracy_score(y_test, y_pred_svm)*100:.2f}%")
print("Confusion Matrix:")
cm_svm = confusion_matrix(y_test, y_pred_svm)
print(cm_svm)
print(f"Weights: {svm_classifier.coef_}")
print(f"Bias: {svm_classifier.intercept_}")

Linear SVM Accuracy: 90.00%
Confusion Matrix:
[[66  2]
 [ 8 24]]
Weights: [[1.60291291 0.97138722]]
Bias: [-0.76862169]


## 6. Export Parameters to C Headers for SiFive RISC-V

In [7]:
# Export SVM weights and bias
weights = svm_classifier.coef_
bias = svm_classifier.intercept_

with open("svm_model.h", "w") as f:
    f.write(f"#define NUM_CLASSES {weights.shape[0]}\n")
    f.write(f"#define NUM_FEATURES {weights.shape[1]}\n")
    f.write("const double weights[NUM_CLASSES][NUM_FEATURES] = {\n")
    for row in weights:
        f.write("   {" + ", ".join(f"{w:.10f}" for w in row) + "},\n")
    f.write("};\n\n")
    f.write("const double bias[NUM_CLASSES] = {" + ", ".join(f"{b:.10f}" for b in bias) + "};\n")

print("✅ Exported SVM model to svm_model.h")

# Export Scaler parameters
mean = sc.mean_
scale = sc.scale_

with open("scaler.h", "w") as f:
    f.write(f"#define NUM_FEATURES {len(mean)}\n\n")
    f.write("const double mean[NUM_FEATURES] = {\n")
    f.write("  " + ", ".join(f"{m:.10f}" for m in mean) + "\n")
    f.write("};\n\n")
    f.write("const double scale[NUM_FEATURES] = {\n")
    f.write("  " + ", ".join(f"{s:.10f}" for s in scale) + "\n")
    f.write("};\n")

print("✅ Exported scaler parameters to scaler.h")

✅ Exported SVM model to svm_model.h
✅ Exported scaler parameters to scaler.h
